# Vaccine Sense - Aplicacao 18

## Coleta e rotulagem do dataset

Traz do InfluxDB as medicoes da caixa termica, voce rotula cada rodada e
salva o dataset que a Aplicacao 19 vai usar para treinar o modelo.

```
ESP32 -> MQTT Broker (local) -> Node-RED (local) -> InfluxDB (cloud) -> Colab
```


## 1. Pacotes


In [ ]:
!pip -q install influxdb3-python pandas pyarrow python-dotenv


## 2. Imports


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from influxdb_client_3 import InfluxDBClient3


## 3. Credenciais

Os mesmos valores que voce configurou no no InfluxDB do Node-RED.


In [ ]:
INFLUX_URL         = "https://us-east-1-1.aws.cloud2.influxdata.com"
INFLUX_TOKEN       = "XXXXX"
INFLUX_ORG         = "XXXXX"
INFLUX_BUCKET      = "XXXXX"
INFLUX_MEASUREMENT = "vaccinesense_raw_2026"

client = InfluxDBClient3(host=INFLUX_URL, token=INFLUX_TOKEN, database=INFLUX_BUCKET)


## 4. O que existe no bucket


In [ ]:
# SQL: lista todas as tabelas/measurements do bucket
tbls = client.query("SHOW TABLES", language="sql")
display(tbls.to_pandas())


In [ ]:
# Lista os campos do measurement do Vaccine Sense
cols = client.query(f'SHOW COLUMNS FROM "{INFLUX_MEASUREMENT}"', language="sql")
display(cols.to_pandas())


## 5. Trazer as medicoes

Ajuste o intervalo para cobrir a sua coleta.


In [ ]:
query = f"""
SELECT
  "time",
  "device",
  "rodada",
  "id",
  "tempInterna",
  "tempExterna",
  "umidade",
  "luz",
  "criticidade",
  "distancia",
  "tempoForaDaFaixa"
FROM "{INFLUX_MEASUREMENT}"
WHERE time >= now() - interval '6 hours'
ORDER BY time
"""

table = client.query(query=query, language="sql")
df = table.to_pandas()
df["time"] = pd.to_datetime(df["time"], utc=True)
df = (df.rename(columns={"time": "timestamp"})
        .set_index("timestamp")
        .sort_index())

print(f"{len(df)} medicoes")
df.tail(15)


## 6. Que rodadas chegaram

Cada vez que voce apertou o botao para iniciar, o firmware abriu uma rodada
nova e numerou. Confira se estao todas aqui e se cada uma tem tempo
suficiente: a 1 Hz, cinco minutos sao cerca de 300 medicoes.

Se voce reiniciou o ESP32 no meio da coleta, a numeracao recomeca do 1 -
olhe as colunas de inicio e fim para perceber.


In [ ]:
resumo = df.groupby("rodada").agg(
    medicoes=("tempInterna", "size"),
    inicio=("tempInterna", lambda s: s.index.min().strftime("%H:%M:%S")),
    fim=("tempInterna", lambda s: s.index.max().strftime("%H:%M:%S")),
)
display(resumo)


## 7. Olhar os dados antes de rotular

Procure nos graficos onde a tampa foi aberta: a luz sobe, a distancia muda e
a temperatura interna comeca a subir depois.


In [ ]:
fig, eixos = plt.subplots(4, 1, figsize=(14, 9), sharex=True)

series = [
    ("tempInterna", "Temp. interna (C)"),
    ("tempExterna", "Temp. externa (C)"),
    ("luz",         "Luz na caixa"),
    ("distancia",   "Distancia (cm)"),
]

for eixo, (coluna, titulo) in zip(eixos, series):
    eixo.plot(df.index, df[coluna], linewidth=1)
    eixo.set_ylabel(titulo)
    eixo.grid(alpha=0.3)

eixos[-1].set_xlabel("horario")
fig.suptitle("Medicoes coletadas")
plt.tight_layout()
plt.show()


## 8. Rotular

Aqui e o trabalho de especialista: **voce sabe** o que fez com a caixa em
cada rodada, e e isso que vira o rotulo.

> O rotulo **nao** vem de nenhum `if` do firmware nem de filtro por horario.
> Ele vem da rodada: o botao numerou, voce anotou o que fez.
> O unico `if` que existe no firmware acende o LED e nao e publicado.

Neste ponto voce ja precisa ter coletado as tres condicoes. Cruze a tabela do
passo 6 com as suas anotacoes e preencha o dicionario - os numeros abaixo sao
exemplo, troque pelos seus.

Rodadas que nao estiverem no dicionario ficam marcadas como `DESCARTAR`.

Confira no grafico do passo 7 se bate: numa rodada de tampa aberta a luz sobe
e a distancia muda. Se nao bater, a anotacao esta trocada.


### Referencia: o que cada rodada devia ter

Se voce seguiu a receita do README, as nove rodadas tem estas assinaturas.
Use a tabela para conferir se a sua anotacao bate com o que os dados mostram.

| # | Rotulo | Temp. interna | Temp. externa | Luz | Distancia |
|---:|---|---|---|---|---|
| 1 | `TRANSPORTE_OK` | ~4 | ~22 | baixa | ~12 |
| 2 | `TRANSPORTE_OK` | ~5 | ~24 | baixa | ~15 |
| 3 | `TRANSPORTE_OK` | ~6 | ~26 | baixa | ~10 |
| 4 | `AMBIENTE_HOSTIL` | ~5 | **~35** | baixa | ~12 |
| 5 | `AMBIENTE_HOSTIL` | ~5,5 | **~37** | baixa | ~14 |
| 6 | `AMBIENTE_HOSTIL` | ~6 | **~40** | baixa | ~12 |
| 7 | `CARGA_EM_PERIGO` | **~9** | **~38** | baixa | ~12 |
| 8 | `CARGA_EM_PERIGO` | **~8,5** | ~25 | **~3000** | **~60** |
| 9 | `CARGA_EM_PERIGO` | **~9,5** | ~24 | **baixa** | **~55** |

Tres conferencias rapidas no grafico do passo 7:

- as rodadas 4, 5 e 6 sobem a **temperatura externa** e mantem a interna boa;
- a rodada 7 sobe as **duas**;
- as rodadas 8 e 9 sobem a **distancia**, mas so a 8 sobe a luz.

Se alguma dessas nao aparecer, a anotacao esta trocada ou a rodada saiu fraca.


In [ ]:
# A chave e o NUMERO da rodada, do jeito que o botao numerou.
# Use a tabela do passo 6 (inicio/fim) e as suas anotacoes de coleta.
SITUACAO = {
    "1": "TRANSPORTE_OK",
    "2": "TRANSPORTE_OK",
    "3": "TRANSPORTE_OK",

    "4": "AMBIENTE_HOSTIL",
    "5": "AMBIENTE_HOSTIL",
    "6": "AMBIENTE_HOSTIL",

    "7": "CARGA_EM_PERIGO",
    "8": "CARGA_EM_PERIGO",
    "9": "CARGA_EM_PERIGO",
}

df["situacao"] = df["rodada"].map(SITUACAO).fillna("DESCARTAR")

display(df["situacao"].value_counts())


## 9. Descartar as transicoes

Os primeiros segundos de cada rodada ainda carregam o efeito do que veio
antes: o sensor estabilizando, o ar se misturando. Fora do dataset.


In [ ]:
SEGUNDOS_DESCARTE = 10

# a 1 Hz, a contagem dentro da rodada equivale a segundos
df["segundo_na_rodada"] = df.groupby("rodada").cumcount()

antes = len(df)
df = df[(df["situacao"] != "DESCARTAR") &
        (df["segundo_na_rodada"] >= SEGUNDOS_DESCARTE)].copy()
df = df.drop(columns=["segundo_na_rodada"])

print(f"{antes} -> {len(df)} medicoes  ({antes - len(df)} descartadas)")


## 10. Conferir o resultado

As tres classes precisam estar presentes e com tamanho parecido. Se
`AMBIENTE_HOSTIL` ficou de fora, o modelo vai confundir dia quente com tampa
aberta.


In [ ]:
display(df["situacao"].value_counts())

cores = {
    "TRANSPORTE_OK":   "tab:blue",
    "AMBIENTE_HOSTIL": "tab:orange",
    "CARGA_EM_PERIGO": "tab:red",
}

fig, eixo = plt.subplots(figsize=(14, 4))
for situacao, grupo in df.groupby("situacao"):
    eixo.scatter(grupo.index, grupo["tempInterna"], s=6,
                 label=situacao, color=cores.get(situacao))

eixo.set_ylabel("Temp. interna (C)")
eixo.set_xlabel("horario")
eixo.set_title("Dataset rotulado")
eixo.grid(alpha=0.3)
eixo.legend()
plt.tight_layout()
plt.show()


## 11. Salvar o dataset


In [ ]:
ARQUIVO = "vaccinesense_dataset.csv"

df.to_csv(ARQUIVO)
print(f"{len(df)} linhas salvas em {ARQUIVO}")
df.head()


In [ ]:
from google.colab import files

files.download(ARQUIVO)


---

## Atencao para a Aplicacao 19

Nem toda coluna do dataset e uma feature. Ao treinar o modelo, **tres
colunas precisam sair**:

| Coluna | Por que |
|---|---|
| `id` | contador sequencial: o modelo separaria as classes pela ordem de coleta, nao pelos sensores |
| `timestamp` | mesma coisa, em outra forma |
| `tempoForaDaFaixa` | e derivado da temperatura interna e so cresce dentro da rodada |

Tambem saem `device`, `rodada` e `situacao` - as duas primeiras sao metadados
e a ultima e o alvo.

As features sao as **seis medicoes**:
`tempInterna`, `tempExterna`, `umidade`, `luz`, `criticidade`, `distancia`.
